# Previsão de Câncer por Tecido

**Autores:** Giulia S. Ferreira, Lucas Candinho e Matheus N. Cunha

`RESUMO`

## Dados da atividade

**Nome da Entrega**: Somente o necessário

**Mural de Quests 4**: Jardim do Palácio

# Introdução

O **TCGA** (_The Cancer Genome Atlas_) foi um dos maiores e mais importantes projetos de pesquisa biomédica realizados sobre o cancêr. O objetivo desse projeto era mapear as alterações genômicas, moleculares e histológicas presentes em diferentes tipos de tumores humanos.

O TCGA coletou e analisou mais de 11 mil pacientes, abrangendo 33 tipos de câncer. Dentro do TCGA, algumas subdivisões foram feitas:
- COAD: Colon Adenocarcinoma;
- READ: Rectum Adenocarcinoma;
- STAD: Stomach Adenocarcinoma;
- HNSC: Head and Neck Squamous Cell Carcinoma;
- ESCA: Esophageal Carcinoma.

Neste notebook, desenvolveremos um modelo preditivo que avaliará qual é o tipo de câncer (dentre os 5 citados) com base nos parâmetros físicos e histológicos da amostra, como será aprofundado adiante.

# Importando Módulos

Para realizar a manipulação de dados em _dataframes_ usaremos o módulo `pandas`. Para um suporte estatístico, autilisaremos o módulo `numpy`. 

Na tentativa de traçar a correlação entre o `target` e as `features`, utilizaremos o `OneHotEncoder` do `sklearn.preprocessing`.

Esse codificador também será utilizado na criação de `Pipelines`. Nessa etapa, importaremos a função `Pipeline` do `sklearn.pipeline` para fazer os `Pipelines` em si. Para os modelos, importaremos os seguintes métodos: `StandardScaler`, `ColumnTransformer`, `PCA`, `RFE`, `LogisticRegression`, `KNeighborsClassifier`, `SVC`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GradientBoostingClassifier`.

.

Além disso, para seção de Validação Cruzada faremos os imports: `from sklearn.model_selection import StratifiedKFold` para dividir os dados mantendo a proporção das classes, e `from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer` importa as métricas de avaliação. `make_scorer` é usado para adaptar as métricas para o formato que o `cross_validate` espera.

In [1]:
# Tratamento de Dados
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder # Usado também na seção das Pipelines 

# Pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer

# Otimização 
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Validação cruzada
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer

c:\Users\lucas25033\OneDrive - ILUM ESCOLA DE CIÊNCIA\Área de Trabalho\Batalha Final\Quest-4-A-batalha-final\ML_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Coleta de Dados

Primeiramente, inserirmos os dados dentro do código, usando do módulo `pandas`.

In [2]:
bacteria = pd.read_csv("Dados/bacteria.unambiguous.decontam.tissue.sample.rpm.relabund.txt", sep="\t", index_col=0).T
metadados = pd.read_csv("Dados/metadata.TCMA.sample.txt", sep="\t")

Podemos, agora, unir os dados em um _dataframe_ só.

In [3]:
df = bacteria.merge(metadados, left_index=True, right_on="bcr_sample_barcode")

Podemos visualizar os nossos dados com o método `<df>.head()`

In [4]:
print(df.head())

     2.0     976.0  1090.0  1117.0    1224.0    1239.0  1297.0   32066.0  \
822  1.0  0.677871     0.0     0.0  0.020165  0.287103     0.0  0.001081   
376  1.0  0.009874     0.0     0.0  0.598349  0.353252     0.0  0.001700   
459  1.0  0.712102     0.0     0.0  0.143843  0.110583     0.0  0.013211   
380  1.0  0.515392     0.0     0.0  0.013983  0.424951     0.0  0.030885   
381  1.0  0.577755     0.0     0.0  0.008072  0.331349     0.0  0.080544   

     40117.0  57723.0  ...  percent_monocyte_infiltration  percent_necrosis  \
822      0.0      0.0  ...                            NaN               5.0   
376      0.0      0.0  ...                            NaN               5.0   
459      0.0      0.0  ...                            NaN               4.0   
380      0.0      0.0  ...                            NaN              11.0   
381      0.0      0.0  ...                            0.0               7.5   

     percent_neutrophil_infiltration  percent_normal_cells  \
822   

# Tratamento de Dados

Nessa seção, realizamos todos os passos de tratamento de dados necessários para utilizarmos esse _dataset_ em nosso modelo.

Antes de tudo, porém, criamos uma cópia do _dataframe_ para mantermos o original intacto.

In [5]:
df_tratado = df.copy(deep=True)

## Removendo colunas com variância 0

Primeiramente removeremos colunas com variância 0, ou seja, colunas que não contribuem para a indução de modelos [1].

In [6]:
num_cols = df.select_dtypes(include=['number'])

variancias = num_cols.var()

colunas_constantes = variancias[variancias == 0].index.tolist()

df_tratado = df_tratado.drop(columns=colunas_constantes)

Perceba que agora temos bem menos colunas (1474 colunas), um fator que facilitará analises posteriores.

In [7]:
print(df_tratado.columns)

Index([                              2.0,                             976.0,
                                  1224.0,                            1239.0,
                                 32066.0,                           74201.0,
                                200795.0,                          200918.0,
                                201174.0,                          203691.0,
       ...
         'percent_monocyte_infiltration',                'percent_necrosis',
       'percent_neutrophil_infiltration',            'percent_normal_cells',
                 'percent_stromal_cells',             'percent_tumor_cells',
                  'percent_tumor_nuclei',                         'project',
                      'HistologicalType',      'ffpe_tumor_slide_submitted'],
      dtype='object', length=1474)


## Análise de Valores Faltantes

Para tratar dos valores faltantes e garantir que o número de valores faltantes, toda coluna que tiver um número de linhas com valores faltantes maior que $437.5$ (que representa 70% do valor total de linhas) será removida do _dataframe_. Inicialmente, vamos analisar quantas delas possuem valores faltantes nessa margem.

In [8]:
linhas_com_nan = df_tratado[df_tratado.isna().any(axis=1)]
soma_nans_por_coluna = linhas_com_nan.isna().sum()
coluna_nans = soma_nans_por_coluna[soma_nans_por_coluna > 437.5].index

soma_nans_por_coluna

2.0                             0
976.0                           0
1224.0                          0
1239.0                          0
32066.0                         0
                             ... 
percent_tumor_cells           320
percent_tumor_nuclei          326
project                         0
HistologicalType              406
ffpe_tumor_slide_submitted    452
Length: 1474, dtype: int64

Agora, vamos remover isso do _dataframe_

In [9]:
df_tratado = df_tratado.drop(columns=coluna_nans)
df_tratado

,2.0,976.0,1224.0,1239.0,32066.0,74201.0,200795.0,200918.0,201174.0,203691.0,...,vessel_used,weight,is_derived_from_ffpe,percent_necrosis,percent_normal_cells,percent_stromal_cells,percent_tumor_cells,percent_tumor_nuclei,project,HistologicalType
822,1.0,0.677871,0.020165,0.287103,0.001081,0.004029,0.0,0.0,0.009751,0.000000,...,Cryovial,702.0,False,5.0,10.0,10.0,75.0,75.0,COAD,NaN
376,1.0,0.009874,0.598349,0.353252,0.001700,0.000000,0.0,0.0,0.036825,0.000000,...,Cryovial,209.0,False,5.0,0.0,15.0,80.0,82.5,COAD,NaN
459,1.0,0.712102,0.143843,0.110583,0.013211,0.016231,0.0,0.0,0.004030,0.000000,...,Cryovial,307.0,False,4.0,2.5,6.0,87.5,90.0,COAD,NaN
380,1.0,0.515392,0.013983,0.424951,0.030885,0.000026,0.0,0.0,0.014728,0.000036,...,Cryovial,400.0,False,11.0,7.5,5.0,76.5,85.0,COAD,NaN
381,1.0,0.577755,0.008072,0.331349,0.080544,0.000000,0.0,0.0,0.002279,0.000000,...,NaN,NaN,True,7.5,0.0,32.5,60.0,80.0,COAD,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,1.0,0.654780,0.180287,0.113244,0.044736,0.000000,0.0,0.0,0.001655,0.001030,...,Cryomold,306.0,False,0.0,10.0,25.0,65.0,65.0,COAD,NaN
529,1.0,0.450869,0.026483,0.111871,0.410650,0.000000,0.0,0.0,0.000128,0.000000,...,Cryomold,200.0,False,1.0,10.0,9.0,80.0,80.0,COAD,NaN
531,1.0,0.854299,0.035233,0.104800,0.000000,0.000000,0.0,0.0,0.005668,0.000000,...,Cryomold,250.0,False,1.0,0.0,0.0,99.0,65.0,COAD,NaN
533,1.0,0.867609,0.010382,0.089753,0.015899,0.002639,0.0,0.0,0.000584,0.000000,...,Cryomold,256.0,False,0.0,0.0,0.0,100.0,65.0,COAD,NaN


## Análise de Colunas com `ints`

O _dataframe_ utilizado foi montado na intenção de comparar a microbiota de diferentes pacientes com diferentes tipos de câncer e comparar com os saudáveis. `< justificativa formal do porquê não iremos usar> `. Como as bactérias estão nomeadas por `ints`, para serem reconhecidas em um outro arquivo, podemos eliminar todas as colunas que possuem um nome como um valor de `int`.

In [10]:
df_tratado = df_tratado.loc[:, df_tratado.columns.map(lambda x: isinstance(x, str))]

## Determinação do `target` e a escolha das `features`

No _dataframe_ escolhido, a coluna `"project"` será o `target`. Essa coluna categórica identifica de qual subprojeto do TCGA aquela amostra se originou. Abaixo, estão os cinco subprojetos do TCGA, que foram definidos previamente.

In [11]:
df_tratado["project"].unique()

array(['COAD', 'READ', 'STAD', 'HNSC', 'ESCA'], dtype=object)

Agora, iremos determinar quais são as 20 colunas com maior valor de correlação o `target`. Para isso, utilizaremos a função `.corr()` do `pandas`. Essa função mede a associação linear entre duas variáveis numéricas

Essa função só calcula correlação entre valores numéricos, de modo que toda coluna que tiver dados do tipo _object_ devam ser codificadas. Para isso, faremos a codificação do tipo `OneHotEncoder`. Esse codificador retorna uma **matriz esparsa** que, apesar de possuir sua eficiência em memória computacional, para os nossos propósitos, dificulta o trabalho com o `pandas`. Para retornar em um `array`, utilizaremos o argumento `sparse_output=False`.

Além disso, para podermos ter um maior controle no tratamento diferenciado da coluna do `TARGET`, isolaremos essa coluna durante essa etapa.

In [12]:
TARGET = "project"

series_to_encode = df_tratado[TARGET].to_frame()

OHE = OneHotEncoder(sparse_output=False)
encoded_target = OHE.fit_transform(series_to_encode)

encoded_df = pd.DataFrame(encoded_target, 
                         columns=OHE.get_feature_names_out([TARGET]),
                         index=df_tratado.index)

df_corr = pd.concat([df_tratado, encoded_df], axis=1)

df_corr = df_corr.drop(TARGET, axis=1)

df_corr = pd.get_dummies(df_corr, drop_first=True)

target_columns = [col for col in df_corr.columns if col.startswith(TARGET)]

correlacoes_target = df_corr.corr()[target_columns].abs()
correlacoes_max = correlacoes_target.max(axis=1).sort_values(ascending=False)

colunas_originais = df_tratado.columns.tolist()
colunas_encoded = [col for col in df_corr.columns if col not in colunas_originais]

correlacoes_filtradas = correlacoes_max.drop(colunas_encoded, errors='ignore')

print("25 features mais correlacionadas com o TARGET:")
print(correlacoes_filtradas.head(25))

features_ordenadas = correlacoes_filtradas.index.tolist()

25 features mais correlacionadas com o TARGET:
intermediate_dimension    0.472776
longest_dimension         0.460243
weight                    0.429243
necrosis_percent          0.410969
shortest_dimension        0.253751
is_ffpe                   0.206250
percent_normal_cells      0.192505
tumor_nuclei_percent      0.179463
percent_tumor_cells       0.153771
sample_code               0.137632
percent_tumor_nuclei      0.126858
sample_type_id            0.097000
percent_necrosis          0.062191
percent_stromal_cells     0.051079
dtype: float64


Note que esse tipo de análise avalia correlações lineares, desprezando eventuais correlações não-lineares que possam haver entre o `target` e as `features`. Com base nesses dados, podemos determinar nossos parâmetros.

Além disso, perceba que as colunas `percent_tumor_nuclei` e a `tumor_nuclei_percent` dizem respeito ao mesmo tipo de análise. Por conta disso, eliminaremos a que obteve a menor correlação linear. As colunas que apresentam análises sobre o tipo de amostra (identificando se é um tecido normal ou cancerígeno) também serão eliminadas visto que todas as amostras do `target` são cancerígenas.

Fora as colunas analisadas por meio da correlação linear, a coluna que avalia se o tumor foi extraído por ressecação cirúrgica, a `ProcurementMethod_Resection`, também será utilizada pela sua capacidade de trazer uma relevância na análise do `target

Dessa forma, nossas `features` são:
- `intermediate_dimension`;
- `longest_dimension`;
- `shortest_dimension`;
- `weight`;
- `necrosis_percent`;
- `is_ffpe`;
- `percent_normal_cells`;
- `tumor_nuclei_percent`;
- `percent_tumor_cells`;
- `percent_stromal_cells`;
- `ProcurementMethod_Resection`

In [13]:
TARGET = "project"

FEATURES = [
    "intermediate_dimension", "longest_dimension", 
    "shortest_dimension", "weight", 
    "necrosis_percent", "is_ffpe", 
    "percent_normal_cells", "tumor_nuclei_percent", 
    "percent_tumor_cells", "percent_stromal_cells"
]

y = df_tratado[TARGET]
X = df_tratado[FEATURES]

# Criação de Pipelines

## Preparação dos Dados

Usaremos as `FEATURES` e o `TARGET`, já definidos na etapa de tratamento de dados, para criarmos os `Pipelines`. Além disso, o código idetificará automaticamente quais das suas *features* são numéricas e quais são categóricas. O que é fundamental para o próximo passo:

O coração da nossa Pipeline de préprocessamento, garantindo que cada coluna receba o tipo de tratamento adequado.


In [14]:
NUMERICAL_FEATURES = X.select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_FEATURES = X.select_dtypes(include=['object', 'bool']).columns.tolist()

X[CATEGORICAL_FEATURES] = X[CATEGORICAL_FEATURES].copy()
for col in CATEGORICAL_FEATURES:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)


print(f"Features Numéricas: {NUMERICAL_FEATURES}")
print(f"Features Categóricas: {CATEGORICAL_FEATURES}")

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, NUMERICAL_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough'
)

Features Numéricas: ['intermediate_dimension', 'longest_dimension', 'shortest_dimension', 'weight', 'necrosis_percent', 'percent_normal_cells', 'tumor_nuclei_percent', 'percent_tumor_cells', 'percent_stromal_cells']
Features Categóricas: ['is_ffpe']


C:\Users\lucas25033\AppData\Local\Temp\ipykernel_18476\3538522158.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[CATEGORICAL_FEATURES] = X[CATEGORICAL_FEATURES].copy()
C:\Users\lucas25033\AppData\Local\Temp\ipykernel_18476\3538522158.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].astype(int)


Em que, 

`numerical_transformer`: É pequena Pipeline que aplica o StandardScaler às colunas numéricas. O StandardScaler padroniza os dados, o que é crucial para modelos como Regressão Logística, KNN e SVM.


`categorical_transformer`: É pequena Pipeline que aplica o OneHotEncoder às colunas categóricas. Este _encoder_ transforma categorias (como 'COAD', 'READ') em colunas binárias (0 ou 1), tornando-as utilizáveis por modelos de ML. O argumento sparse_output=False garante que a saída seja um array denso, mais fácil de manipular.


`ColumnTransformer`: Combina os transformadores. Ele aplica o numerical_transformer apenas às colunas listadas em NUMERICAL_FEATURES e o categorical_transformer apenas às colunas em CATEGORICAL_FEATURES. Isso evita que o scaler seja aplicado em dados categóricos e vice-versa.

 O preprocessor é a primeira etapa de todas as pipelines. Ele garante que os dados estejam limpos e padronizados antes de qualquer outra técnica mais avançada ser aplicada.

## Definição dos Modelos Base e Criação das Pipelines

Este bloco define os modelos que serão testados e os parâmetros iniciais para as técnicas de seleção de features:


`models`: Um dicionário que lista todos os modelos que serão testados, incluindo os novos (Random Forest e Gradient Boosting). Note que o SVC (SVM) recebeu o parâmetro probability=True para que possa calcular a métrica AUC-ROC, que depende de probabilidades.

**Criação das Pipelines**: O loop for cria uma Pipeline para cada modelo. Cada pipeline tem duas etapas:

- `preprocessor`: O ColumnTransformer definido no passo anterior.
- `classifier`: O modelo de machine learning (ex: Regressão Logística).
Isso garante que o pré-processamento seja aplicado somente aos dados de treino em cada fold da validação cruzada, evitando data leakage.


In [15]:
# Definição dos Modelos Base 
base_models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=33),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(kernel='linear', probability=True, random_state=33),
    "DecisionTree": DecisionTreeClassifier(random_state=33),
    "RandomForest": RandomForestClassifier(random_state=33),
    "GradientBoosting": GradientBoostingClassifier(random_state=33)
}

#  Criação das Pipelines 
pipelines = {}

for name, model in base_models.items():
    # Pipeline Baseline (Apenas Pré-processamento e Modelo)
    pipelines[name] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

print(f"Total de {len(pipelines)} pipelines criadas para otimização.")

Total de 6 pipelines criadas para otimização.


# Otimização com `Optuna`

## Por que otimizar com o `Optuna`?

Cada um dos 24 `Pipelines` descritos possuem hiperparâmetros distintos e independentes: 
- `n_neighbors`, `weights`, `metric` no KNN;
- `C` e `kernel` no SVM;
- `max_depth`, `min_samples_split` na árvore de decisão;
- `n_estimators_`, `max_features` e `max_depth` nos modelos _ensemble_;
- `n_components` do PCA;
- `n_features_to_select` do RFE.

Dada a complexidade e heterogeneidade desses `Pipelines`, o espaço de hiperparâmetros é não-linear, de alta dimensionalidade e **interdependente**. Isso torna métodos de busca exaustiva ou aleatória, como o _grid search_ e o _random search_, ineficientes. O `Optuna`, por outro lado, emprega um algoritmo de _Tree-structured Parzen Estimator_ (TPE), que desenvolve probabilisticamente a relação entre os hiperparâmetros e a métrica de desempenho, direcionando itarativamente a busca para regiões mais promissoras do espaço. Essa abordagem leva a uma redução do número de avaliações necessárias para atingir resultados aceitáveis, de modo que seja compatível com as restrições computacionais do grupo.

Além disso, o conjunto de dados apresenta um moderado desbalanço nas classes do _target_, sendo a menor frequência da classe `READ`, de 7.8% de ocorrência. Isso reforça a necessidade de um método de otimização adaptativo, que permite ajustar modelos sensíveis à distribuição dos dados. 

## Criação das Funções Objetivo

Primeiramente, criamos o nosso método de validação cruzada (para a otimização) e o nosso avaliador de desempenho.

Note que o número de _splits_ está justificado na Validação Cruzada geral, mas também o utilizaremos aqui.

Adotamos também o _F1 score weighted_ que calcula a média harmônica entre acurácia e revocação - proporção de itens positivos que foram corretamente identificados por um modelo - refletindo de maneira mais justa o desempenho, evitando que classes majoritárias dominem a métrica.

In [16]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=33)
scorer = "f1_weighted"

Para o KNN, iremos realizar a busca com os seguintes parâmetros:

- Número de vizinhos de 3 - 70, a abrangencia é escolhida visto que os dados são ruidosos e mal-comportados, um número de vizinhos alto torna o modelo menos sensível ao ruido;
- Pesos uniforme e baseado em distância, visto que não podemos ter certeza de como os dados influenciam uns aos outros;
- Metrica de distância Manhattan e Minkowski, que são melhores em dimensões maiores (ao contrário da Euclidiana).


In [ ]:
def objective_knn(trial):
    pipe = pipelines['KNN'].set_params()
    
    n_neighbors = trial.suggest_int("n_neighbors", 1, 20)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    metric = trial.suggest_categorical("metric", ["manhattan", "minkowski"])
    pipe.set_params(classifier__n_neighbors=n_neighbors,
                    classifier__weights=weights,
                    classifier__metric=metric)
    
    return cross_val_score(pipe, X, y, cv=cv, scoring=scorer).mean()

Para o SVM, a busca será feita com:
- 

In [ ]:
def objective_svm(trial):
    pipe = pipelines['SVM'].set_params()
    
    C = trial.suggest_float("C", 1e-3, 1e3, log=True)
    kernel = trial.suggest_categorical("kernel", ["rbf", "poly"])
    gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
    
    pipe.set_params(classifier__C=C, classifier__kernel=kernel, classifier__gamma=gamma)
    
    if kernel == "poly":
        degree = trial.suggest_int("degree", 2, 5) 
        pipe.set_params(classifier__degree=degree)
    
    return cross_val_score(pipe, X, y, cv=cv, scoring=scorer).mean()


In [ ]:



def objective_decision_tree(trial):
    pipe = pipelines['DecisionTree'].set_params()
    
    max_depth = trial.suggest_int("max_depth", 2, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"])
    pipe.set_params(classifier__max_depth=max_depth,
                    classifier__min_samples_split=min_samples_split,
                    classifier__criterion=criterion)
    
    return cross_val_score(pipe, X, y, cv=cv, scoring=scorer).mean()


def objective_random_forest(trial):
    pipe = pipelines['RandomForest'].set_params()
    
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 30)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    pipe.set_params(classifier__n_estimators=n_estimators,
                    classifier__max_depth=max_depth,
                    classifier__max_features=max_features,
                    classifier__min_samples_split=min_samples_split)
    
    return cross_val_score(pipe, X, y, cv=cv, scoring=scorer).mean()


def objective_gradient_boosting(trial):
    pipe = pipelines['GradientBoosting'].set_params()
    
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-3, 1.0)
    max_depth = trial.suggest_int("max_depth", 2, 10)
    subsample = trial.suggest_uniform("subsample", 0.5, 1.0)
    pipe.set_params(classifier__n_estimators=n_estimators,
                    classifier__learning_rate=learning_rate,
                    classifier__max_depth=max_depth,
                    classifier__subsample=subsample)
    
    return cross_val_score(pipe, X, y, cv=cv, scoring=scorer).mean()


Agrupamos, então, as funções em um dicionário.

In [18]:
opt_functions = {
    "KNN": objective_knn,
    "SVM": objective_svm,
    "DecisionTree": objective_decision_tree,
    "RandomForest": objective_random_forest,
    "GradientBoosting": objective_gradient_boosting
}

## Realização da Otimização

Podemos, agora, realizar um estudo para cada uma das funções objetivo criadas.

In [19]:
resultados = {}

for name, objective in opt_functions.items():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50, n_jobs=-1)
    resultados[name] = {"best_params": study.best_params, "best_score": study.best_value}

[I 2025-11-02 17:02:56,080] A new study created in memory with name: no-name-e304e8c4-743b-4474-89e3-4581a0017c84


[I 2025-11-02 17:03:00,204] Trial 0 finished with value: 0.6191971421310787 and parameters: {'n_neighbors': 20, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 0 with value: 0.6191971421310787.
[I 2025-11-02 17:03:00,810] Trial 6 finished with value: 0.6151562757494482 and parameters: {'n_neighbors': 15, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 0 with value: 0.6191971421310787.
[I 2025-11-02 17:03:00,868] Trial 1 finished with value: 0.6218623173392106 and parameters: {'n_neighbors': 15, 'weights': 'uniform', 'metric': 'minkowski'}. Best is trial 1 with value: 0.6218623173392106.
[I 2025-11-02 17:03:00,943] Trial 4 finished with value: 0.6190549850648241 and parameters: {'n_neighbors': 10, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 1 with value: 0.6218623173392106.
[I 2025-11-02 17:03:01,074] Trial 12 finished with value: 0.6090791888971675 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'manhattan'}. Best is trial 1 w

Vemos, então, os melhores modelos.

In [20]:
for model_name, res in resultados.items():
    print(f"\n {model_name}")
    print("Melhores parâmetros:", res["best_params"])
    print("Melhor f1_weighted:", round(res["best_score"], 4))



 KNN
Melhores parâmetros: {'n_neighbors': 11, 'weights': 'uniform', 'metric': 'minkowski'}
Melhor f1_weighted: 0.6273

 SVM
Melhores parâmetros: {'C': 0.5091084962280645, 'kernel': 'rbf', 'gamma': 'auto'}
Melhor f1_weighted: 0.6407

 DecisionTree
Melhores parâmetros: {'max_depth': 20, 'min_samples_split': 12, 'criterion': 'gini'}
Melhor f1_weighted: 0.7192

 RandomForest
Melhores parâmetros: {'n_estimators': 53, 'max_depth': 29, 'max_features': None, 'min_samples_split': 9}
Melhor f1_weighted: 0.7197

 GradientBoosting
Melhores parâmetros: {'n_estimators': 124, 'learning_rate': 0.8160974991470018, 'max_depth': 3, 'subsample': 0.5755426742356791}
Melhor f1_weighted: 0.7218


# Validação Cruzada

A validação cruzada é uma técnica essencial para obter uma estimativa de desempenho do modelo que seja robusta e imparcial. 
Ao invés de dividir o *dataset* em apenas um conjunto de treino e um de teste, o que pode introduzir um viés de seleção (se a divisão de teste for atipicamente fácil) ou uma alta variância na estimativa (se a divisão de teste for atipicamente difícil), a Validação Cruzada K-Fold mitiga esses riscos.
Ela divide o *dataset* em $K$ subconjuntos (neste caso, $K=10$). O modelo é treinado $K$ vezes, usando $K-1$ subconjuntos para treino e o subconjunto restante para teste. 
O resultado final é a média das $K$ execuções, o que fornece uma estimativa muito mais confiável do desempenho do modelo em dados não vistos, reduzindo a dependência de uma única partição aleatória. [4]

 **Por que K=10$?**

Adotamos o valor de $K=10$ folds, que é o padrão na literatura e *machine learning* [5], pois este valor representa um equilíbrio ideal entre:
- `Viés (Bias)`: Um $K$ muito pequeno, aumentando o viés. 
- `Variância (Varience)`: Um $K$ muito grande (ex: $K=N$, onde N é o número de features) aumenta a variância e o custo computacional.

O valor de $K=19$ é amplamente aceito por fornecer uma estimativa de desempenho com baixo viés e variância moderada, sendo um compromisso eficiente entre a precisãoda estimativa e o tempo de processamento.

A validação cruzada estratificada é crucial para problemas de classificação (como o que estamos trabalhando, que classifica o tipo de cancêr). Ela garante que a proporção das classes do *target* (os tipos de cancêr) seja mantida em cada um dos $K$ subconjuntos. Isso é vital para evitar que a avaliaçãp seja distorcida pela distribuição desigual das classes. [6]

Em resumo, a Validação Cruzada Final é o que confirma o desempenho generalizável do modelo otmizado.


In [21]:
# Definindo o melhor modelo otimizado 
# valores arbitrários por enquanto

melhor_pipeline = pipelines['LogReg'] # Exemplo: A melhor foi a Regressão Logística

# Os hiperparâmetros focam na regularização e complexidade do modelo
melhores_hiperparametros = {
    'classifier__C': 0.05,             # Exemplo de hiperparâmetro de regularização (LogReg/SVM)
    'classifier__penalty': 'l2',       # Exemplo de tipo de regularização (LogReg)
    'classifier__max_depth': 10        # Exemplo de hiperparâmetro de complexidade (RandomForest/DecisionTree)
}

# Aplicar os melhores hiperparâmetros à pipeline
melhor_pipeline.set_params(**melhores_hiperparametros)

# Definição das Métricas de Validação Cruzada
# Classificação multiclasse

scoring = {
    'Acurácia': make_scorer(accuracy_score),
    'Precisão (Macro)': make_scorer(precision_score, average='macro', zero_division=0),
    'Recall (Macro)': make_scorer(recall_score, average='macro', zero_division=0),
    'F1-Score (Macro)': make_scorer(f1_score, average='macro', zero_division=0),
    'AUC-ROC (OVR)': make_scorer(roc_auc_score, needs_proba=True, multi_class='ovr', average='macro', zero_division=0)
}

# VC final 
# Usamos 10 folds para uma avaliação robusta
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=33)


ValueError: Invalid parameter 'max_depth' for estimator LogisticRegression(C=0.05, max_iter=1000, random_state=33). Valid parameters are: ['C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'multi_class', 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start'].

Estamos  trabalhando na Previsão de Câncer por Tecido (TCGA), onde o objetivo é classificar uma amostra de tecido em um dos vários tipos de câncer (COAD, READ, STAD, HNSC, ESCA, etc.), logo usaremos a classificação multiclasse.
- Múltiplas Classes[9]: Seu target (project) não é binário (sim/não, 0/1), mas sim uma variável categórica com múltiplas categorias mutuamente exclusivas (os diferentes tipos de câncer).

Obs. Sempre que o target tem mais de duas categorias mutuamente exclusivas, o problema é, por definição, um problema de Classificação Multiclasse.

O uso da classificação multiclasse reflete diretamente nas métricas que escolhemos: 

- Precisão, Recall, F1-Score: `avarege=macro` -> em problemas multiclasse, o modelo calcula a métrica para cada classe individualmente. O average='macro' calcula a média dessas métricas, dando o mesmo peso a cada tipo de câncer. Isso é crucial para evitar que o desempenho seja dominado pelas classes mais frequentes. [7]

- AUC-ROC: `multi-class=ovr` -> A AUC-ROC foi originalmente projetada para problemas binários. Para adaptá-la ao multiclasse, o método One-vs-Rest (OVR) é usado. Ele treina um classificador binário para cada classe (ex: COAD vs. Não-COAD) e depois calcula a média das AUCs de todos esses classificadores.[8]

**Em resumo:** Usamos a classificação multiclasse porque o dataset exige que o modelo escolha entre mais de duas opções de câncer. Os parâmetros average='macro' e multi_class='ovr' garantem que essa avaliação seja feita de forma justa e robusta, considerando o desempenho em cada tipo de câncer.


# Treino do Modelo

# Teste e Avaliação do Modelo

# Explicação do Modelo Utilizado

# Conclusão

# XKCD Relevante

![img](https://xkcd.com/1349/)

`Imagem: Shouldn't Be Hard (XKCD) disponível em https://xkcd.com/1349/`

# Referências

## _Dataset_ Utilizado

## Gerais

 1. Notebook \"ATP-203 8.1 - Seleção de atributos\" do professor Dr. Daniel R. Cassar;
 2. Notebook \"ATP-203 7.1 - Dados Sintéticos e Pipeline\" do professor Dr. Daniel R. Cassar;
 3. Scikit-learn Developers. Pipeline and Composite Estimators. Scikit-learn Documentation. Link: https://scikit-learn.org/stable/modules/compose.html;
 4. Notebook \"ATP-203 6.0 - Validação Cruzada e Otimização de Hiperparâmetros\" do Professor Dr. Daniel R. Cassar;
 5. Livro , seção 7.10 - Hastie, T., Tibshirani, R., & Friedman, J. (2009). The Elements of Statistical Learning: Data Mining, Inference, and Prediction (2nd ed.). Springer.
 Disponível em: https://hastie.su.domains/ElemStatLearn/
 6. Guia Validação Cruzada K-Fold - https://www.datacamp.com/pt/tutorial/k-fold-cross-validation
 7. Scikit-learn Developers. Classification metrics. Scikit-learn Documentation. Link: https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics
 8. Scikit-learn Developers. OneVsRestClassifier. Scikit-learn Documentation. Link: https://scikit-learn.org/stable/modules/generated/sklearn.multiclass.OneVsRestClassifier.html 
 9. Artigo sobre avaliação de modelos - Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. Information Processing & Management, 45(4), 427–437.
Link: https://doi.org/10.1016/j.ipm.2009.03.002
 